In [106]:
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx

from utils import data as ld
from utils.stats import attach_entropy_zscore, flag_sse

In [4]:
columns = [
    "window_id", "window_idx", "wn_mid_date", "cluster_id", "cluster_size",
    "cluster_n_datazones", "dz_health_board", "dz_local_authority",
    "datazone", "age_band", "sex", "dz_simd_quintile",
    "who_voc", "clade", "dz_7d_test_positivity", "cluster_duration_days",
    'wn_positive_tests', 'wn_prop_sequenced',  'dz_cum_sequences',
    'dz_cum_positive_tests', 'dz_cum_prop_sequenced', 'dz_cum_incidence_per_capita',
]

df_raw = ld.load_analysis_columns(columns, add_policy=True).to_pandas()

In [5]:
df_raw = attach_entropy_zscore(
    df=df_raw,
    cluster_col="cluster_id",
    category_col="sex",
    window_col="window_id",
    prefix="sex",
)

df_raw = attach_entropy_zscore(
    df=df_raw,
    cluster_col="cluster_id",
    category_col="age_band",
    window_col="window_id",
    prefix="age",
)

df_raw = attach_entropy_zscore(
    df=df_raw,
    cluster_col="cluster_id",
    category_col="dz_simd_quintile",
    window_col="window_id",
    prefix="simd",
)

df_raw = attach_entropy_zscore(
    df=df_raw,
    cluster_col="cluster_id",
    category_col="dz_health_board",
    window_col="window_id",
    prefix="health_board",
)

In [6]:
def join_top(values: pd.Series, n: int = 5) -> str:
    counts = values.dropna().astype(str).value_counts()
    return "; ".join(f"{name} ({count})" for name, count in counts.head(n).items())

def safe_mode(values: pd.Series):
    if values.empty:
        return np.nan
    return values.mode().iloc[0]

entropy_cols = [c for c in df_raw.columns if "entropy" in c]

agg = {
    "cluster_size": ("cluster_size", "first"),
    "duration_days": ("cluster_duration_days", "first"),
    "cluster_n_datazones": ("cluster_n_datazones", "first"),
    "first_collection_date": ("collection_date", "min"),
    "last_collection_date": ("collection_date", "max"),
    "who_voc": ("who_voc", "first"),
    "clade": ("clade", "first"),
    "pango_lineage": ("pango_lineage", "first"),
    "policy_period": ("policy_period", safe_mode),
    "policy_periods": ("policy_period", "nunique"),
    "health_board": ("dz_health_board", safe_mode),
    "health_boards": ("dz_health_board", "nunique"),
    "local_authority": ("dz_local_authority", safe_mode),
    "local_authorities": ("dz_local_authority", "nunique"),
    "dz_7d_test_positivity": ("dz_7d_test_positivity", "mean"),
    "dz_cum_sequences": ("dz_cum_sequences", "mean"),
    "dz_cum_incidence_per_capita": ("dz_cum_incidence_per_capita", "mean"),
    "dz_cum_positive_tests": ("dz_cum_positive_tests", "mean"),
    "dz_cum_prop_sequenced": ("dz_cum_prop_sequenced", "mean"),
    "wn_positive_tests": ("wn_positive_tests", "first"),
    "wn_prop_sequenced": ("wn_prop_sequenced", "first"),
    **{col: (col, "first") for col in entropy_cols},
}

cluster_table = (
    df_raw.groupby(["cluster_id", "window_id", "window_idx", "wn_mid_date"], as_index=False)
    .agg(**agg)
)

In [12]:
seq_nodes = (
    df_raw[["sequence_id", "cluster_id", "window_id", "window_idx"]]
    .drop_duplicates()
    .sort_values(["sequence_id", "window_idx"])
)

edge_map: dict[tuple[str, str, object, object, int, int], set[str]] = defaultdict(set)

for sequence_id, group in seq_nodes.groupby("sequence_id", sort=False):
    records = list(
        group[["cluster_id", "window_id", "window_idx"]]
        .itertuples(index=False, name=None)
    )

    for i, (source_node, source_window, source_idx) in enumerate(records):
        for target_node, target_window, target_idx in records[i + 1:]:
            delta = target_idx - source_idx

            if delta == 1 and source_node != target_node:
                edge_key = (
                    source_node,
                    target_node,
                    source_window,
                    target_window,
                    source_idx,
                    target_idx,
                )
                edge_map[edge_key].add(sequence_id)

            elif delta > 1:
                break

edge_table = pd.DataFrame(
    [
        {
            "source": source,
            "target": target,
            "source_window_id": source_window,
            "target_window_id": target_window,
            "source_window_idx": source_idx,
            "target_window_idx": target_idx,
            "n_shared_sequences": len(shared),
        }
        for (
            source,
            target,
            source_window,
            target_window,
            source_idx,
            target_idx,
        ), shared in edge_map.items()
    ]
).sort_values(
    [
        "source_window_idx",
        "target_window_idx",
        "n_shared_sequences",
        "source",
        "target",
    ],
    ascending=[True, True, False, True, True],
    ignore_index=True,
)

G_raw = nx.DiGraph()
for row in cluster_table.itertuples(index=False):
    G_raw.add_node(row.cluster_id)
for row in edge_table.itertuples(index=False):
    G_raw.add_edge(row.source, row.target, weight=row.n_shared_sequences)

components = sorted(nx.weakly_connected_components(G_raw), key=lambda nodes: (-len(nodes), sorted(nodes)[0]))
component_map = {
    node: f"AM{component_idx:05d}"
    for component_idx, nodes in enumerate(components, start=1)
    for node in nodes
}
cluster_table["meta_cluster_id"] = cluster_table["cluster_id"].map(component_map)

print(f"Graph nodes: {G_raw.number_of_nodes():,}")
print(f"Graph edges: {G_raw.number_of_edges():,}")
print(f"Connected components / meta-clusters: {len(components):,}")

Graph nodes: 193,160
Graph edges: 148,184
Connected components / meta-clusters: 51,773


In [13]:
node_summary = (
    cluster_table.groupby("meta_cluster_id", as_index=False)
    .agg(
        n_clusters=("cluster_id", "nunique"),
        n_windows=("window_id", "nunique"),
        first_window_mid_date=("wn_mid_date", "min"),
        last_window_mid_date=("wn_mid_date", "max"),
        max_cluster_size=("cluster_size", "max"),
        max_cluster_n_datazones=("cluster_n_datazones", "max"),
    )
)

seq_meta_long = df_raw[[
        "sequence_id", "collection_date", "cluster_id",
        "pango_lineage",  "who_voc", "clade",
    ]].merge(
    cluster_table[["cluster_id", "meta_cluster_id"]],
    on="cluster_id", how="left"
)

seq_meta_long["n_candidate_meta_clusters"] = (
    seq_meta_long.groupby("sequence_id")["meta_cluster_id"].transform("nunique")
)

seq_meta_long["ambiguous_meta_assignment"] = seq_meta_long["n_candidate_meta_clusters"] > 1

sequence_summary = (
    seq_meta_long.groupby("meta_cluster_id", as_index=False)
    .agg(
        n_sequences=("sequence_id", "nunique"),
        first_collection_date=("collection_date", "min"),
        last_collection_date=("collection_date", "max"),
        pango_lineage=("pango_lineage", "first"),
        clade=("clade", "first"),
        who_voc=("who_voc", "first"),
        ambiguous_sequence_assignments=("ambiguous_meta_assignment", "sum"),
    )
)
meta_summary = node_summary.merge(
    sequence_summary, on="meta_cluster_id", how="left"
)

meta_summary = meta_summary.sort_values(
    ["n_sequences", "n_clusters"], ascending=[False, False]
)
meta_summary["duration_days"] = (
    meta_summary["last_collection_date"] -
    meta_summary["first_collection_date"]
).dt.days

In [107]:
sse_odor = flag_sse(seq_meta_long)

In [91]:
# calculate in-degree
in_degree = (
    edge_table.groupby("target")
    .size()
    .rename("in_degree")
    .reset_index()
    .rename(columns={"target": "cluster_id"})
)

weighted_in = (
    edge_table.groupby("target")["n_shared_sequences"]
    .sum()
    .rename("in_strength")
    .reset_index()
    .rename(columns={"target": "cluster_id"})
)

node_stats = (
    cluster_table
    .merge(in_degree, on="cluster_id", how="left")
    .merge(weighted_in, on="cluster_id", how="left")
)

In [92]:
def downstream_entropy_fast(edge_df, source_col, weight_col):
    df = edge_df[[source_col, weight_col]].copy()
    df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce")
    df = df.dropna()
    df = df[df[weight_col] > 0]

    codes, nodes = pd.factorize(df[source_col], sort=False)
    order = np.argsort(codes, kind="mergesort")

    codes = codes[order]
    w = df[weight_col].to_numpy(float)[order]

    starts = np.r_[0, np.flatnonzero(np.diff(codes)) + 1]
    counts = np.diff(np.r_[starts, len(w)])

    strength = np.add.reduceat(w, starts)
    wlogw = np.add.reduceat(w * np.log(w), starts)

    entropy = np.log(strength) - (wlogw / strength)
    entropy[counts <= 1] = np.nan

    entropy_norm = entropy / np.log(counts)
    entropy_norm[counts <= 1] = np.nan

    dominant_frac = np.maximum.reduceat(w, starts) / strength

    return pd.DataFrame({
        "out_degree": counts,
        "out_strength": strength,
        "downstream_entropy": entropy,
        "downstream_entropy_norm": entropy_norm,
        "effective_successors": np.exp(entropy),
        "dominant_successor_frac": dominant_frac,
    }, index=pd.Index(nodes, name=source_col))


def downstream_entropy_null_fast(
    edge_df,
    source_col,
    weight_col,
    n_perm=199,
    seed=42,
    metric_col="downstream_entropy_norm",
):
    df = edge_df[[source_col, weight_col]].copy()
    df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce")
    df = df.dropna()
    df = df[df[weight_col] > 0]

    observed = downstream_entropy_fast(df, source_col, weight_col)
    obs = observed[metric_col].to_numpy()
    obs_finite = np.isfinite(obs)

    codes, nodes = pd.factorize(df[source_col], sort=False)
    order = np.argsort(codes, kind="mergesort")

    codes = codes[order]
    weights = df[weight_col].to_numpy(float)[order]

    starts = np.r_[0, np.flatnonzero(np.diff(codes)) + 1]
    counts = np.diff(np.r_[starts, len(weights)])

    rng = np.random.default_rng(seed)

    n = np.zeros(len(nodes), dtype=int)
    mean = np.zeros(len(nodes), dtype=float)
    m2 = np.zeros(len(nodes), dtype=float)
    ge_obs = np.zeros(len(nodes), dtype=int)
    le_obs = np.zeros(len(nodes), dtype=int)

    for _ in range(n_perm):
        w = rng.permutation(weights)

        strength = np.add.reduceat(w, starts)
        wlogw = np.add.reduceat(w * np.log(w), starts)

        entropy = np.log(strength) - (wlogw / strength)
        entropy[counts <= 1] = np.nan

        vals = entropy / np.log(counts)
        vals[counts <= 1] = np.nan

        finite = np.isfinite(vals)

        n[finite] += 1
        delta = vals[finite] - mean[finite]
        mean[finite] += delta / n[finite]
        m2[finite] += delta * (vals[finite] - mean[finite])

        comparable = finite & obs_finite
        ge_obs[comparable] += vals[comparable] >= obs[comparable]
        le_obs[comparable] += vals[comparable] <= obs[comparable]

    sd = np.full(len(nodes), np.nan)
    ok = n > 1
    sd[ok] = np.sqrt(m2[ok] / (n[ok] - 1))

    z = (obs - mean) / np.where(sd == 0, np.nan, sd)

    p_high = np.full(len(nodes), np.nan)
    p_low = np.full(len(nodes), np.nan)
    ok = n > 0
    p_high[ok] = (ge_obs[ok] + 1) / (n[ok] + 1)
    p_low[ok] = (le_obs[ok] + 1) / (n[ok] + 1)

    null = pd.DataFrame({
        f"{metric_col}_null_mean": mean,
        f"{metric_col}_null_sd": sd,
        f"{metric_col}_z": z,
        f"{metric_col}_p_high": p_high,
        f"{metric_col}_p_low": p_low,
    }, index=pd.Index(nodes, name=source_col))

    return observed.join(null)

In [93]:
null_df = downstream_entropy_null_fast(
    edge_df=edge_table,
    source_col="source",
    weight_col="n_shared_sequences",
    n_perm=1000,
)

In [94]:
node_stats = node_stats.merge(
    null_df.reset_index().rename(columns={"source": "cluster_id"}),
    on="cluster_id",
    how="left",
)

In [95]:
# ===================================================================
# Utility functions
# ===================================================================

def safe_divide(numerator, denominator):
    """
    Divide two columns/arrays safely.
    Returns NaN where denominator is 0 or missing.
    """
    denominator_safe = denominator.replace(0, np.nan)
    out = numerator / denominator_safe
    return out.replace([np.inf, -np.inf], np.nan)


def log1p_ratio(numerator, denominator):
    """
    Stable log ratio:
        log((1 + numerator) / (1 + denominator))

    Useful when zero values are meaningful, e.g. no upstream input.
    Note: this is a log ratio, not log(numerator - denominator).
    """
    return np.log1p(numerator) - np.log1p(denominator)


def add_window_percentile(df, col, groupby_cols="window_idx"):
    """
    Percentile rank of a feature within each temporal window.
    groupby_cols can be a string or list for stratified ranking
    (e.g. ["window_idx", "lifecycle"] to compare like-for-like
    lifecycle phases).
    """
    return df.groupby(groupby_cols)[col].rank(pct=True, method="average")


def add_window_zscore(df, col, groupby_cols="window_idx"):
    """
    Z-score of a feature within each temporal window.
    Uses ddof=0 so single-member groups return NaN rather than error.
    """
    grouped = df.groupby(groupby_cols)[col]
    mean = grouped.transform("mean")
    std  = grouped.transform("std", ddof=0).replace(0, np.nan)
    return (df[col] - mean) / std


def mean_with_count(df, cols, skipna=False):
    """
    Row-wise mean across cols with a companion count of valid components.
    skipna=False propagates NaN if any component is missing, making
    scores comparable (all rows use the same number of components).
    Returns (score_series, n_valid_series).
    """
    sub = df[[c for c in cols if c in df.columns]]
    score   = sub.mean(axis=1, skipna=skipna)
    n_valid = sub.notna().sum(axis=1)
    return score, n_valid


# ===================================================================
# 0. Lifecycle classification and list-column parsing
# ===================================================================


# -- Lifecycle flags --------------------------------------------------
node_stats["birth"]        = node_stats["in_strength"].eq(0)
node_stats["birth_like"]        = node_stats["in_strength"].le(1)
node_stats["death"]        = node_stats["out_degree"].eq(0)
node_stats["continuation"] = node_stats["in_degree"].gt(0) & node_stats["out_degree"].gt(0)
node_stats["branching"]    = node_stats["out_degree"].gt(1)
node_stats["merging"]      = node_stats["in_degree"].gt(1)

# Mutually exclusive lifecycle label for stratified ranking
node_stats["lifecycle"] = np.select(
    [node_stats["birth"], node_stats["death"], node_stats["continuation"]],
    ["birth",             "death",             "continuation"],
    default="unknown",
)

# Compound SSE-relevant lifecycle states
node_stats["simple_chain"]    = ( node_stats["continuation"] &
                                 ~node_stats["branching"] &
                                 ~node_stats["merging"])

node_stats["birth_branching"] = ( node_stats["birth"] &
                                   node_stats["branching"])

node_stats["expansion"]       = ( node_stats["continuation"] &
                                   node_stats["branching"] &
                                  ~node_stats["merging"])

node_stats["hub"]             = ( node_stats["branching"] &
                                   node_stats["merging"])

node_stats["sink"]            = ( node_stats["merging"] &
                                   node_stats["death"])

node_stats["isolated"]        = ( node_stats["birth"] &
                                   node_stats["death"])

# -- Censoring flags --------------------------------------------------
# Nodes at the temporal boundary of the dataset or a VOC epoch should
# be excluded from birth/death-based analyses.
first_window = node_stats["window_idx"].min()
last_window  = node_stats["window_idx"].max()

node_stats["left_censored"]  = node_stats["birth"]  & node_stats["window_idx"].eq(first_window)
node_stats["right_censored"] = node_stats["death"]   & node_stats["window_idx"].eq(last_window)

first_per_voc = node_stats.groupby("who_voc")["window_idx"].transform("min")
last_per_voc  = node_stats.groupby("who_voc")["window_idx"].transform("max")

node_stats["epoch_left_censored"]  = node_stats["birth"]  & node_stats["window_idx"].eq(first_per_voc)
node_stats["epoch_right_censored"] = node_stats["death"]  & node_stats["window_idx"].eq(last_per_voc)


# ===================================================================
# 1. Basic directed degree imbalance
# ===================================================================

node_stats["degree_imbalance"] = (
    node_stats["out_degree"] - node_stats["in_degree"]
)


node_stats["strength_imbalance"] = (
    node_stats["out_strength"] - node_stats["in_strength"]
)

# log1p_ratio: log((1 + w_out) / (1 + w_in))
node_stats["log_strength_ratio"] = log1p_ratio(
    node_stats["out_strength"],
    node_stats["in_strength"],
)

node_stats["strength_ratio"] = safe_divide(
    node_stats["out_strength"],
    node_stats["in_strength"],
)


# ===================================================================
# 2. Local amplification / novelty relative to upstream input
# ===================================================================
# NOTE: for birth nodes (in_strength == 0) these metrics
# resolve to a function of cluster_size alone.  Interpret within
# lifecycle strata (see percentile section below).

node_stats["upstream_novelty_proxy"] = (
    node_stats["cluster_size"] / (1 + node_stats["in_strength"])
)

node_stats["net_amplification"] = (
    node_stats["cluster_size"] - node_stats["in_strength"]
)


# ===================================================================
# 3. Downstream dissemination / persistence
# ===================================================================

node_stats["downstream_retention_ratio"] = safe_divide(
    node_stats["out_strength"],
    node_stats["cluster_size"],
)

node_stats["log_downstream_retention_ratio"] = log1p_ratio(
    node_stats["out_strength"],
    node_stats["cluster_size"],
)

# downstream_expansion_proxy: combines retention (what fraction leaves)
# with fan-out (how many branches).  High when a cluster both retains
# many sequences downstream AND splits into multiple chains.
node_stats["downstream_expansion_proxy"] = (
    node_stats["downstream_retention_ratio"] * node_stats["out_degree"]
)


# ===================================================================
# 4. Geographic spread
# ===================================================================

node_stats["datazone_density"] = safe_divide(
    node_stats["cluster_n_datazones"],
    node_stats["cluster_size"],
)

node_stats["mean_sequences_per_datazone"] = safe_divide(
    node_stats["cluster_size"],
    node_stats["cluster_n_datazones"],
)


node_stats["health_board_density"] = safe_divide(
    node_stats["health_boards"],
    node_stats["cluster_size"],
)

node_stats["mean_sequences_per_health_board"] = safe_divide(
    node_stats["cluster_size"],
    node_stats["health_boards"],
)

node_stats["local_authority_density"] = safe_divide(
    node_stats["local_authorities"],
    node_stats["cluster_size"],
)

node_stats["mean_sequences_per_local_authority"] = safe_divide(
    node_stats["cluster_size"],
    node_stats["local_authorities"],
)

node_stats = node_stats.copy()


# ===================================================================
# 5. Within-window percentile ranks
# ===================================================================
# Two sets of percentiles:
#
#   (a) window_idx only — how extreme is this node among ALL
#       contemporaneous nodes?  Useful for the composite SSE scores.
#
#   (b) [window_idx, lifecycle] — how extreme within its lifecycle
#       phase?  Corrects for the systematic inflation of amplification
#       metrics at birth nodes (where in_strength == 0).

percentile_cols = [
    "cluster_size",
    "out_degree",
    "out_strength",
    "degree_imbalance",
    "strength_imbalance",
    "log_strength_ratio",
    "net_amplification",
    "upstream_novelty_proxy",
    "downstream_retention_ratio",
    "downstream_expansion_proxy",
    "cluster_n_datazones",
    "datazone_density",
]

for col in percentile_cols:
    if col not in node_stats.columns:
        continue
    # (a) window-wide
    node_stats[f"{col}_pct_window"] = add_window_percentile(
        node_stats, col, groupby_cols="window_idx"
    )
    # (b) lifecycle-stratified
    node_stats[f"{col}_pct_window_lifecycle"] = add_window_percentile(
        node_stats, col, groupby_cols=["window_idx", "lifecycle"]
    )


# ===================================================================
# 6. Within-window z-scores
# ===================================================================

zscore_cols = [
    "cluster_size",
    "in_strength",
    "out_strength",
    "net_amplification",
    "upstream_novelty_proxy",
    "downstream_expansion_proxy",
    "cluster_n_datazones",
]

for col in zscore_cols:
    if col not in node_stats.columns:
        continue
    node_stats[f"{col}_z_window"] = add_window_zscore(
        node_stats, col, groupby_cols="window_idx"
    )


# ===================================================================
# 7. Composite SSE scores
# ===================================================================
# skipna=False ensures all rows use the same number of components.
# Inspect the companion *_n columns to verify completeness.
#
# Three deliberately separate dimensions — do not collapse further
# without epidemiological justification:
#
#   core_amplification_score   — sudden local growth
#   onward_dissemination_score — downstream fan-out
#   mixing_score               — population bridging (entropy-based)

# -- Core amplification -----------------------------------------------
core_components = [
    "cluster_size_pct_window",
    "net_amplification_pct_window",
    "upstream_novelty_proxy_pct_window",
]

(
    node_stats["core_amplification_score"],
    node_stats["core_amplification_n"],
) = mean_with_count(node_stats, core_components, skipna=False)

# -- Onward dissemination ---------------------------------------------
onward_components = [
    "out_degree_pct_window",
    "weighted_out_degree_pct_window",
    "downstream_expansion_proxy_pct_window",
    "downstream_retention_ratio_pct_window",
]

(
    node_stats["onward_dissemination_score"],
    node_stats["onward_dissemination_n"],
) = mean_with_count(node_stats, onward_components, skipna=False)

# -- Population mixing / bridging -------------------------------------
# Entropy z-scores are already null-model-corrected (observed vs
# random draws of same size), so they are directly comparable across
# cluster sizes and windows without further normalisation.
# Positive values → more diverse than expected; 1.96 ≈ p < 0.05.
mixing_components = [
    "simd_entropy_z",
    "health_board_entropy_z",
    "age_entropy_z",
    "sex_entropy_z",
]

(
    node_stats["mixing_score"],
    node_stats["mixing_score_n"],
) = mean_with_count(node_stats, mixing_components, skipna=False)

In [99]:
def categorise_sse_nodes(
    df,
    high_q=0.90,
    very_high_q=0.95,
    entropy_high=0.65,
    entropy_low=0.35,
    dominant_high=0.75,
):
    df = df.copy()

    def has(col):
        return col in df.columns

    def bool_col(col):
        if has(col):
            return df[col].fillna(False).astype(bool)
        return pd.Series(False, index=df.index)

    def high_col(col, q=high_q):
        """
        For percentile columns, compare directly to q.
        For raw score columns, use empirical quantile.
        """
        if not has(col):
            return pd.Series(False, index=df.index)

        x = pd.to_numeric(df[col], errors="coerce")

        if col.endswith("_pct_window") or col.endswith("_pct_window_lifecycle"):
            return x >= q

        cutoff = x.quantile(q)
        return x >= cutoff

    def low_col(col, q=1 - high_q):
        if not has(col):
            return pd.Series(False, index=df.index)

        x = pd.to_numeric(df[col], errors="coerce")

        if col.endswith("_pct_window") or col.endswith("_pct_window_lifecycle"):
            return x <= q

        cutoff = x.quantile(q)
        return x <= cutoff

    birth_like = bool_col("birth_like")
    death = bool_col("death")
    continuation = bool_col("continuation")
    merging = bool_col("merging")
    isolated = bool_col("isolated")
    left_censored = bool_col("left_censored") | bool_col("epoch_left_censored")
    right_censored = bool_col("right_censored") | bool_col("epoch_right_censored")

    out_degree = pd.to_numeric(df.get("out_degree", 0), errors="coerce").fillna(0)
    in_degree = pd.to_numeric(df.get("in_degree", 0), errors="coerce").fillna(0)
    out_strength = pd.to_numeric(df.get("out_strength", 0), errors="coerce").fillna(0)
    in_strength = pd.to_numeric(df.get("in_strength", 0), errors="coerce").fillna(0)

    entropy_norm = pd.to_numeric(df.get("downstream_entropy_norm", np.nan), errors="coerce")
    entropy_z = pd.to_numeric(df.get("downstream_entropy_norm_z", np.nan), errors="coerce")
    entropy_p_high = pd.to_numeric(df.get("downstream_entropy_norm_p_high", np.nan), errors="coerce")
    entropy_p_low = pd.to_numeric(df.get("downstream_entropy_norm_p_low", np.nan), errors="coerce")
    dominant_frac = pd.to_numeric(df.get("dominant_successor_frac", np.nan), errors="coerce")

    high_entropy = (
        (entropy_norm >= entropy_high)
        | ((entropy_z >= 1.96) & (entropy_p_high <= 0.05))
    )

    low_entropy = (
        (entropy_norm <= entropy_low)
        | (dominant_frac >= dominant_high)
        | ((entropy_z <= -1.96) & (entropy_p_low <= 0.05))
    )

    high_core_amp = high_col("core_amplification_score", high_q)
    very_large = high_col("cluster_size_pct_window_lifecycle", very_high_q)
    high_novelty = high_col("upstream_novelty_proxy_pct_window_lifecycle", high_q)
    high_net_amp = high_col("net_amplification_pct_window_lifecycle", high_q)
    high_downstream_expansion = high_col("downstream_expansion_proxy_pct_window_lifecycle", high_q)
    high_out_strength = high_col("out_strength_pct_window_lifecycle", high_q)

    spatially_broad = (
        high_col("cluster_n_datazones_pct_window_lifecycle", high_q)
        | high_col("datazone_density_pct_window_lifecycle", high_q)
        | high_col("mixing_score", high_q)
        | (pd.to_numeric(df.get("health_board_entropy_z", np.nan), errors="coerce") >= 1.96)
    )

    # First: is this node plausibly an SSE-like amplification?
    df["sse_candidate"] = (
        high_core_amp
        | (very_large & (high_novelty | high_net_amp))
        | (high_novelty & high_downstream_expansion)
        | (
            pd.to_numeric(df.get("cluster_size_z_window", np.nan), errors="coerce").ge(2)
            & pd.to_numeric(df.get("net_amplification_z_window", np.nan), errors="coerce").ge(1)
        )
    )

    # Origin / role of the node.
    df["sse_role"] = np.select(
        [
            birth_like & high_novelty,
            continuation & (in_strength > 0) & (out_strength > 0) & high_net_amp,
            merging & (in_degree >= 2) & (out_strength > 0),
            (in_strength > 0) & (out_strength == 0),
            isolated,
        ],
        [
            "putative_birth",
            "relay_amplifier",
            "merged_relay",
            "terminal_sink",
            "isolated_burst",
        ],
        default="unclear_origin",
    )

    # Onward dynamics after the candidate node.
    df["sse_onward_dynamic"] = np.select(
        [
            out_strength == 0,
            death | ((out_degree <= 1) & low_col("downstream_expansion_proxy_pct_window_lifecycle", 0.25)),
            (out_degree >= 1) & low_entropy,
            (out_degree >= 2) & high_entropy & high_downstream_expansion & spatially_broad,
            (out_degree >= 2) & high_entropy & high_downstream_expansion,
            (out_degree >= 2) & high_entropy,
            (out_degree >= 2) & high_out_strength,
        ],
        [
            "no_observed_onward_spread",
            "contained_burst",
            "single_dominant_chain",
            "diffuse_spatial_broadcaster",
            "multi_branch_expander",
            "multi_branch_seeder",
            "high_volume_onward_spread",
        ],
        default="weak_or_ambiguous_onward_spread",
    )

    # Final phenotype label.
    df["sse_category"] = np.where(
        ~df["sse_candidate"],
        "not_sse_like",
        df["sse_role"] + "__" + df["sse_onward_dynamic"],
    )

    df["sse_censoring_note"] = np.select(
        [
            left_censored & right_censored,
            left_censored,
            right_censored,
        ],
        [
            "both_left_and_right_censored",
            "left_censored_origin_uncertain",
            "right_censored_onward_uncertain",
        ],
        default="not_censored",
    )

    return df

In [100]:
node_stats = categorise_sse_nodes(node_stats)

In [102]:
print(node_stats["sse_category"].value_counts(dropna=False))

sse_category
not_sse_like                                    178939
relay_amplifier__single_dominant_chain            3776
terminal_sink__no_observed_onward_spread          1892
putative_birth__single_dominant_chain             1602
unclear_origin__single_dominant_chain             1533
relay_amplifier__contained_burst                  1381
unclear_origin__contained_burst                    956
merged_relay__contained_burst                      520
putative_birth__contained_burst                    449
relay_amplifier__diffuse_spatial_broadcaster       408
merged_relay__single_dominant_chain                276
putative_birth__no_observed_onward_spread          241
putative_birth__multi_branch_expander              235
relay_amplifier__multi_branch_expander             228
unclear_origin__multi_branch_expander              146
relay_amplifier__high_volume_onward_spread         130
relay_amplifier__multi_branch_seeder               120
merged_relay__multi_branch_seeder                   